# 01 Data Quality and Exploratory Workforce Analysis

Professional People Analytics notebook for Employee Attrition Prediction & Workforce Intelligence.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "raw" / "WA_Fn-UseC_-HR-Employee-Attrition.csv").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from the repository or one of its subfolders.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('D:/Project/Data Science/employee_attrition_prediction_&_workforce_intelligence')

## Objective

Validate the raw IBM HR Analytics dataset before any modeling. This notebook checks row and column contracts, missing values, duplicates, target imbalance, leakage columns, categorical cardinality, and numeric ranges.

In [2]:
import pandas as pd

from src.data import (
    categorical_cardinality,
    data_dictionary_frame,
    load_raw_data,
    numeric_range_summary,
    run_data_quality_audit,
    validate_raw_dataset,
)

df = load_raw_data()
validate_raw_dataset(df)
audit = run_data_quality_audit(df)
audit

DataQualityResult(row_count=1470, column_count=35, missing_values=0, duplicate_rows=0, target_distribution={'No': 1233, 'Yes': 237}, attrition_rate=0.16122448979591836, constant_columns=['EmployeeCount', 'Over18', 'StandardHours'], identifier_columns=['EmployeeNumber'])

In [3]:
quality_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "missing_values",
            "duplicate_rows",
            "attrition_no",
            "attrition_yes",
            "attrition_rate",
        ],
        "value": [
            audit.row_count,
            audit.column_count,
            audit.missing_values,
            audit.duplicate_rows,
            audit.target_distribution["No"],
            audit.target_distribution["Yes"],
            round(audit.attrition_rate, 4),
        ],
    }
)
quality_summary

,metric,value
0,rows,1470.0000
1,columns,35.0000
2,missing_values,0.0000
3,duplicate_rows,0.0000
4,attrition_no,1233.0000
5,attrition_yes,237.0000
6,attrition_rate,0.1612


## Data Dictionary and Leakage Roles

In [4]:
data_dictionary_frame(df)

,column,dtype,role,missing_values,unique_values,description
0,Age,int64,numeric_feature,0,43,Employee age in years.
1,Attrition,object,target,0,2,Target label indicating whether the employee l...
2,BusinessTravel,object,categorical_feature,0,3,Business travel frequency.
3,DailyRate,int64,numeric_feature,0,886,Daily compensation rate recorded in the source...
4,Department,object,categorical_feature,0,3,Employee department.
5,DistanceFromHome,int64,numeric_feature,0,29,Distance from home to workplace.
6,Education,int64,numeric_feature,0,5,Education level coded from 1 to 5.
7,EducationField,object,categorical_feature,0,6,Field of education.
8,EmployeeCount,int64,constant_excluded,0,1,Constant source column; excluded from modeling.
9,EmployeeNumber,int64,identifier_excluded,0,1470,Employee identifier; retained for reporting bu...


## Categorical Cardinality

In [5]:
categorical_cardinality(df)

,column,unique_values,top_value
5,JobRole,9,Sales Executive
3,EducationField,6,Life Sciences
6,MaritalStatus,3,Married
2,Department,3,Research & Development
1,BusinessTravel,3,Travel_Rarely
0,Attrition,2,No
4,Gender,2,Male
8,OverTime,2,No
7,Over18,1,Y


## Numeric Range Review

In [6]:
numeric_range_summary(df).sort_values('column')

,column,min,median,mean,max
0,Age,18.0,36.0,36.923810,60.0
1,DailyRate,102.0,802.0,802.485714,1499.0
2,DistanceFromHome,1.0,7.0,9.192517,29.0
3,Education,1.0,3.0,2.912925,5.0
4,EmployeeCount,1.0,1.0,1.000000,1.0
5,EmployeeNumber,1.0,1020.5,1024.865306,2068.0
6,EnvironmentSatisfaction,1.0,3.0,2.721769,4.0
7,HourlyRate,30.0,66.0,65.891156,100.0
8,JobInvolvement,1.0,3.0,2.729932,4.0
9,JobLevel,1.0,2.0,2.063946,5.0


## Key EDA Tables

In [7]:
def attrition_rate_by(column: str) -> pd.DataFrame:
    return (
        df.assign(attrition_flag=(df["Attrition"] == "Yes").astype(int))
        .groupby(column)
        .agg(employees=("attrition_flag", "size"), attrition_rate=("attrition_flag", "mean"))
        .reset_index()
        .sort_values("attrition_rate", ascending=False)
    )

attrition_rate_by("Department")

,Department,employees,attrition_rate
2,Sales,446,0.206278
0,Human Resources,63,0.190476
1,Research & Development,961,0.138398


In [8]:
attrition_rate_by('OverTime')

,OverTime,employees,attrition_rate
1,Yes,416,0.305288
0,No,1054,0.104364


In [9]:
attrition_rate_by('BusinessTravel')

,BusinessTravel,employees,attrition_rate
1,Travel_Frequently,277,0.249097
2,Travel_Rarely,1043,0.149569
0,Non-Travel,150,0.080000
